In [ ]:
# Data Transformation
"""


Objective:
This assignment aims to equip you with practical skills in data preprocessing, feature engineering, and feature selection techniques, which are crucial for building efficient machine learning models. You will work with a provided dataset to apply various techniques such as scaling, encoding, and feature selection methods including isolation forest and PPS score analysis.
Dataset:
Given "Adult" dataset, which predicts whether income exceeds $50K/yr based on census data.
Tasks:
1.	Handle missing values as per the best practices (imputation, removal, etc.).
●	Apply scaling techniques to numerical features:
a.	Standard Scaling   b. Min-Max Scaling
●	Discuss the scenarios where each scaling technique is preferred and why.
2. Encoding Techniques:
●	Apply One-Hot Encoding to categorical variables with less than 5 categories.
●	Use Label Encoding for categorical variables. Data Exploration and Preprocessing:
●	Load the dataset and conduct basic data exploration (summary statistics, missing values, data types).
●	les with more than 5 categories.
●	Discuss the pros and cons of One-Hot Encoding and Label Encoding.
3. Feature Engineering:
●	Create at least 2 new features that could be beneficial for the model. Explain the rationale behind your choices.
●	Apply a transformation (e.g., log transformation) to at least one skewed numerical feature and justify your choice.

"""


In [38]:
import pandas as pd
import numpy as  np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

In [39]:
df = pd.read_csv('Assi_9_adult_with_headers.csv')

In [40]:
print("Dataset Shape:", df.shape)

Dataset Shape: (32561, 15)


In [41]:
print("\nFirst 5 rows:")
print(df.head())


First 5 rows:
   age          workclass  fnlwgt   education  education_num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital_status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital_gain  capital_loss  hours_per_week  native_country  income  
0          2174             0              40   Unite

In [42]:
print("\nData Types:")
print(df.dtypes)


Data Types:
age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object


In [43]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


In [46]:
import numpy as np

# Handling Missing value
df_clean = df.replace('?', np.nan)

In [49]:
# Fill categorical missing values with mode
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
print("Missing values after cleaning:", df_clean.isnull().sum().sum())

Missing values after cleaning: 0


In [50]:
#Scaling Technique


In [51]:
numerical_cols = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

# Standard Scaling
standard_scaler = StandardScaler()
df_standard = df_clean.copy()
df_standard[numerical_cols] = standard_scaler.fit_transform(df_clean[numerical_cols])

# Min-Max Scaling
minmax_scaler = MinMaxScaler()
df_minmax = df_clean.copy()
df_minmax[numerical_cols] = minmax_scaler.fit_transform(df_clean[numerical_cols])

print("Standard Scaling - mean=0, std=1")
print(df_standard[numerical_cols].head())
print("\nMin-Max Scaling - range [0,1]")
print(df_minmax[numerical_cols].head())

print("\nWhen to use Standard Scaling:")
print("- Data follows normal distribution")
print("- Algorithms: PCA, SVM, Linear Regression, Neural Networks")

print("\nWhen to use Min-Max Scaling:")
print("- Data does NOT follow normal distribution")
print("- Algorithms: KNN, K-Means, Neural Networks with sigmoid")

Standard Scaling - mean=0, std=1
        age    fnlwgt  education_num  capital_gain  capital_loss  \
0  0.030671 -1.063611       1.134739      0.148453      -0.21666   
1  0.837109 -1.008707       1.134739     -0.145920      -0.21666   
2 -0.042642  0.245079      -0.420060     -0.145920      -0.21666   
3  1.057047  0.425801      -1.197459     -0.145920      -0.21666   
4 -0.775768  1.408176       1.134739     -0.145920      -0.21666   

   hours_per_week  
0       -0.035429  
1       -2.222153  
2       -0.035429  
3       -0.035429  
4       -0.035429  

Min-Max Scaling - range [0,1]
        age    fnlwgt  education_num  capital_gain  capital_loss  \
0  0.301370  0.044302       0.800000       0.02174           0.0   
1  0.452055  0.048238       0.800000       0.00000           0.0   
2  0.287671  0.138113       0.533333       0.00000           0.0   
3  0.493151  0.151068       0.400000       0.00000           0.0   
4  0.150685  0.221488       0.800000       0.00000           0.0   

In [52]:
# Encoding Technique


In [53]:
categorical_cols = ['workclass', 'education', 'marital_status', 'occupation', 
                    'relationship', 'race', 'sex', 'native_country']

In [55]:
# One-Hot Encoding for columns with <5 categories
onehot_cols = [col for col in categorical_cols if df_clean[col].nunique() < 5]
print("One-Hot Encoding columns (<5 categories):", onehot_cols)

df_encoded = df_clean.copy()
for col in onehot_cols:
    dummies = pd.get_dummies(df_clean[col], prefix=col, drop_first=True)
    df_encoded = pd.concat([df_encoded, dummies], axis=1)
    df_encoded.drop(col, axis=1, inplace=True)



One-Hot Encoding columns (<5 categories): ['sex']


In [56]:
# Label Encoding for columns with >=5 categories
label_cols = [col for col in categorical_cols if df_clean[col].nunique() >= 5]
print("\nLabel Encoding columns (>=5 categories):", label_cols)

for col in label_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_clean[col])



Label Encoding columns (>=5 categories): ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'native_country']


In [57]:
# Encode target
df_encoded['income'] = df_clean['income'].map({'<=50K': 0, '>50K': 1})

print("\nPros and Cons of One-Hot Encoding:")
print("Pros: No ordinal assumption, works with linear models")
print("Cons: Increases dimensions, memory intensive")

print("\nPros and Cons of Label Encoding:")
print("Pros: Single column, memory efficient, works with tree models")
print("Cons: Implies ordinal relationship, can mislead linear models")


Pros and Cons of One-Hot Encoding:
Pros: No ordinal assumption, works with linear models
Cons: Increases dimensions, memory intensive

Pros and Cons of Label Encoding:
Pros: Single column, memory efficient, works with tree models
Cons: Implies ordinal relationship, can mislead linear models


In [58]:
# 3 Feature Engineering


In [59]:
df_fe = df_clean.copy()


In [60]:
# Feature 1: Age Group (categorizes age into meaningful brackets)
def age_group(age):
    if age < 25: return 0      # Young
    elif age < 40: return 1    # Adult
    elif age < 60: return 2    # Middle_Aged
    else: return 3             # Senior

df_fe['age_group'] = df_fe['age'].apply(age_group)
print("Feature 1 Created: age_group")
print("Rationale: Income pattern differs by age - young earn less, middle-aged earn peak")


Feature 1 Created: age_group
Rationale: Income pattern differs by age - young earn less, middle-aged earn peak


In [61]:
# Feature 2: Total Capital (combines gain and loss)
df_fe['total_capital'] = df_fe['capital_gain'] + df_fe['capital_loss']
print("\nFeature 2 Created: total_capital")
print("Rationale: Captures overall investment activity better than separate features")



Feature 2 Created: total_capital
Rationale: Captures overall investment activity better than separate features


In [62]:
# Feature 3: Work Hours Squared (captures non-linear effect)
df_fe['hours_squared'] = df_fe['hours_per_week'] ** 2
print("\nFeature 3 Created: hours_squared")
print("Rationale: Working very long hours may have diminishing returns on income")


Feature 3 Created: hours_squared
Rationale: Working very long hours may have diminishing returns on income


In [63]:
#  Log Transformation for Skewed Feature



In [64]:
print("\nSkewness before transformation:")
print(f"capital_gain skewness: {df_fe['capital_gain'].skew():.2f}")


Skewness before transformation:
capital_gain skewness: 11.95


In [65]:
# Apply log1p transformation (adds 1 to handle zeros)
df_fe['capital_gain_log'] = np.log1p(df_fe['capital_gain'])


In [66]:
print("\nSkewness after log transformation:")
print(f"capital_gain_log skewness: {df_fe['capital_gain_log'].skew():.2f}")

print("\nWhy log transformation?")
print("- Reduces right skewness")
print("- Makes distribution closer to normal")
print("- Helps linear models perform better")
print("- log1p handles zero values appropriately")


Skewness after log transformation:
capital_gain_log skewness: 3.10

Why log transformation?
- Reduces right skewness
- Makes distribution closer to normal
- Helps linear models perform better
- log1p handles zero values appropriately
